# FlowCast (ICLR 2026) on INSAT-3S — VIS + WV

Drives the four-stage FlowCast pipeline on a Colab T4 GPU.

1. **Stage 1** — encode INSAT TIFFs with FlowCast's released SEVIR VAE (`b-rbmp/flowcast-cfm-sevir`)
2. **Stage 2** — sanity-check the VAE roundtrip (decision gate: PSNR ≥ 25 dB)
3. **Stage 3** — train the Earthformer-UNet I-CFM model (σ = 0.01, AdamW + cosine + EMA + FP16)
4. **Stage 4** — 48-frame forecast via 4 autoregressive blocks of 12, 8-sample pixel-space ensemble, threshold-based meteorological metrics (CRPS, CSI-M, FSS-M-P16, HSS-M, FAR-M)

Set the runtime to **GPU** (Runtime → Change runtime type → T4 GPU) before starting.

## Setup — mount Drive, cd, install deps (run after every fresh runtime)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/ISRO/ISRO A.1'   # <-- edit if your Drive path differs
%cd "$PROJECT_DIR"
!ls

In [ ]:
!pip install -q -r requirements.txt
!pip install -q imagecodecs huggingface_hub
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## VIS — Stage 1: encode TIFFs with the FlowCast VAE

Downloads the SEVIR autoencoder checkpoint once, then writes one `.pt` per frame to `vis/latents_fc/`.
Norm stats are computed on the first run and saved back into `vis/config_fc.yaml`.

In [ ]:
!python -m flowcast.encode_latents --config vis/config_fc.yaml

## VIS — Stage 2: VAE sanity check (decision gate)

The FlowCast VAE was trained on SEVIR radar, not INSAT — this is the load-bearing OOD check.

- **PSNR ≥ 25 dB**: proceed.
- **PSNR 20–25 dB**: proceed but expect a ceiling.
- **PSNR < 20 dB**: stop and decide (fall back to SD-VAE pipeline).

In [ ]:
from IPython.display import Image, display
!python -m flowcast.sanity_check --config vis/config_fc.yaml
display(Image('vis/outputs_fc/sanity_grid.png'))

## VIS — Stage 3: train the Earthformer-UNet I-CFM model

Paper-spec settings (T4-scaled): hidden=192, depth=4 cuboid blocks per stage, 4 attention heads,
lag=13 / lead=12, σ=0.01, AdamW lr=5e-4, cosine + 1% warmup, EMA decay 0.999, FP16.
Roughly 12–24 h on T4 across 2–3 sessions; the checkpoint is saved on every best-val improvement.

In [ ]:
!python -m flowcast.train_flow --config vis/config_fc.yaml

## VIS — Stage 4: 48-frame forecast (single + 8-sample ensemble)

In [ ]:
!python -m flowcast.forecast_flow --config vis/config_fc.yaml --samples 1
!python -m flowcast.forecast_flow --config vis/config_fc.yaml --samples 8

In [ ]:
import glob, json
from IPython.display import Image, display

for p in sorted(glob.glob('vis/outputs_fc/flow_forecast_*_grid.png'))[-6:]:
    print(p); display(Image(p))

for p in sorted(glob.glob('vis/outputs_fc/flow_forecast_*_metrics.json')):
    m = json.load(open(p))
    print('\n', p, ' samples=', m.get('samples'), ' medoid_idx=', m.get('medoid_idx'))
    print('  thresholds:', m.get('thresholds'))
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        print(f'  [{mode}]')
        for k, v in (smry or {}).items():
            print(f'    {k:>10}: {v:.4f}')
    if m.get('crps_mean') is not None:
        print(f'  [   crps]')
        print(f'    {"CRPS":>10}: {m["crps_mean"]:.4f}')


## VIS — Blur reduction (staged A → B + 3-way ensemble)

The forecast cells above produce **pixmean / medoid / latmean** trajectories side-by-side once `--samples > 1`. Below: fine-tune the VAE decoder on real frames (Strategy A), then on (predicted_latent → GT) pairs (Strategy B). Plan reference: `~/.claude/plans/s-whats-further-action-steady-clock.md`.

Phase numbers below match the plan.

### Phase 2 — Strategy A: decoder fine-tune on real (encode → decode) pairs

~45 min on T4. Freezes encoder, unfreezes decoder + post_quant_conv, optimises `0.1·MSE + 1.0·LPIPS-VGG`.
Saves checkpoint to `vis/checkpoints_fc/vae_decoder_ft_A.pt`.

In [ ]:
!pip install -q lpips==0.1.4

In [ ]:
!python -m flowcast.finetune_vae --config vis/config_fc.yaml

### Phase 3 — enable A + validate

Uncomments `vae.decoder_ckpt: vis/checkpoints_fc/vae_decoder_ft_A.pt` in the VIS config, then re-runs sanity_check (expect roundtrip PSNR +6 to +10 dB over baseline) and forecast (expect +0.5 to +1.5 dB on medoid).

In [ ]:
# enable A's decoder for inference
import re, pathlib
cfg_path = pathlib.Path('vis/config_fc.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)#\s*decoder_ckpt:\s*vis/checkpoints_fc/vae_decoder_ft_A\.pt.*$',
    r'\1decoder_ckpt: vis/checkpoints_fc/vae_decoder_ft_A.pt',
    text, count=1, flags=re.MULTILINE)
if new_text == text:
    new_text = re.sub(
        r'^(\s*)decoder_ckpt:\s*vis/checkpoints_fc/vae_decoder_ft_B\.pt.*$',
        r'\1decoder_ckpt: vis/checkpoints_fc/vae_decoder_ft_A.pt',
        text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
print('vae block now:')
for line in new_text.splitlines():
    if 'vae' in line.lower() or 'decoder_ckpt' in line:
        print(' ', line)


In [ ]:
!python -m flowcast.sanity_check --config vis/config_fc.yaml --num 30
!python -m flowcast.forecast_flow --config vis/config_fc.yaml --samples 8

### Phase 4 — build (predicted_latent → GT) pair dataset for Strategy B

~80 min on T4. Runs the CFM forecaster on every train-only day × 4 rollouts; saves predicted latents paired with GT metadata. Hard leakage asserts: no val/test UTCs touch the dataset.

In [ ]:
!python -m flowcast.build_pred_dataset --config vis/config_fc.yaml

### Phase 5 — Strategy B: decoder fine-tune on (pred_latent → GT) pairs

~100 min on T4. Decoder learns to map forecaster-output latents directly to sharp GT, compensating for both VAE OOD blur and forecaster mean-drift in one step. Inner train/val_holdout split is by-day (last 10% of training-pair days held out for early stop). Saves to `vis/checkpoints_fc/vae_decoder_ft_B.pt`.

In [ ]:
!python -m flowcast.finetune_vae_pred --config vis/config_fc.yaml

### Phase 6 — swap config to B + validate

Swaps `decoder_ckpt` from `_ft_A.pt` to `_ft_B.pt`. Note: sanity_check PSNR will DROP 1–3 dB vs A — the decoder is now specialised for forecaster latents, not encoder-mode latents. This is expected, not a regression. The signal that matters is the forecast metrics, especially medoid PSNR/SSIM/LPIPS.

In [ ]:
# swap A -> B
import re, pathlib
cfg_path = pathlib.Path('vis/config_fc.yaml')
text = cfg_path.read_text()
new_text = re.sub(
    r'^(\s*)decoder_ckpt:\s*vis/checkpoints_fc/vae_decoder_ft_A\.pt.*$',
    r'\1decoder_ckpt: vis/checkpoints_fc/vae_decoder_ft_B.pt',
    text, count=1, flags=re.MULTILINE)
cfg_path.write_text(new_text)
for line in new_text.splitlines():
    if 'decoder_ckpt' in line:
        print(' ', line)

In [ ]:
!python -m flowcast.sanity_check --config vis/config_fc.yaml --num 30
!python -m flowcast.forecast_flow --config vis/config_fc.yaml --samples 8

In [ ]:
# compare A vs B vs baseline (run the metrics-display cell above too)
import glob, json
all_runs = sorted(glob.glob('vis/outputs_fc/flow_forecast_*_metrics.json'))
print(f'{"file":<60s} {"mode":<10s} {"psnr":>7s} {"ssim":>7s} {"CSI_M":>7s}')
for p in all_runs[-3:]:
    m = json.load(open(p))
    name = p.split('/')[-1]
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        psnr = smry.get('psnr', 0); ssim = smry.get('ssim', 0); csi = smry.get('CSI_M', 0)
        print(f'{name:<60s} {mode:<10s} {psnr:7.3f} {ssim:7.3f} {csi:7.3f}')
    if m.get('crps_mean') is not None:
        print(f'{"":<60s} {"crps":<10s} {m["crps_mean"]:7.4f}')

## WV — Stage 1 → 4 (same recipe, no day/night handling)

Don't kick this off until the VIS Stage-2 sanity check has cleared the decision gate above. WV is
easier (continuous 24 h signal) but pays cross-block drift over 4 rollout blocks.

In [ ]:
!python -m flowcast.encode_latents --config wv/config_fc.yaml
!python -m flowcast.sanity_check  --config wv/config_fc.yaml

In [ ]:
from IPython.display import Image, display
display(Image('wv/outputs_fc/sanity_grid.png'))

In [ ]:
!python -m flowcast.train_flow --config wv/config_fc.yaml

In [ ]:
!python -m flowcast.forecast_flow --config wv/config_fc.yaml --samples 1
!python -m flowcast.forecast_flow --config wv/config_fc.yaml --samples 8

In [ ]:
import glob, json
from IPython.display import Image, display

for p in sorted(glob.glob('wv/outputs_fc/flow_forecast_*_grid.png'))[-6:]:
    print(p); display(Image(p))

for p in sorted(glob.glob('wv/outputs_fc/flow_forecast_*_metrics.json')):
    m = json.load(open(p))
    print('\n', p, ' samples=', m.get('samples'), ' medoid_idx=', m.get('medoid_idx'))
    print('  thresholds:', m.get('thresholds'))
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        print(f'  [{mode}]')
        for k, v in (smry or {}).items():
            print(f'    {k:>10}: {v:.4f}')
    if m.get('crps_mean') is not None:
        print(f'  [   crps]')
        print(f'    {"CRPS":>10}: {m["crps_mean"]:.4f}')


## (Optional) Snapshot a channel's results to a shareable folder + zip

In [ ]:
import os, shutil, glob, json, zipfile, datetime as dt
import yaml

CH = 'vis'   # change to 'wv' for the WV snapshot
TS = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
SNAP = f'{CH}/results_snapshot_fc_{TS}'
os.makedirs(SNAP, exist_ok=True)

for p in sorted(glob.glob(f'{CH}/outputs_fc/*')):
    if os.path.isfile(p):
        shutil.copy2(p, os.path.join(SNAP, os.path.basename(p)))
for src in (f'{CH}/checkpoints_fc/best_flow.pt',
            f'{CH}/checkpoints_fc/flow_history.json',
            f'{CH}/checkpoints_fc/vae_decoder_ft_A.pt',
            f'{CH}/checkpoints_fc/vae_decoder_ft_A_history.json',
            f'{CH}/checkpoints_fc/vae_decoder_ft_B.pt',
            f'{CH}/checkpoints_fc/vae_decoder_ft_B_history.json'):
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(SNAP, os.path.basename(src)))
shutil.copy2(f'{CH}/config_fc.yaml', os.path.join(SNAP, 'config_fc.yaml'))

lines = [f'FlowCast {CH.upper()} snapshot — {TS}', '']
cfg = yaml.safe_load(open(f'{CH}/config_fc.yaml'))
lines += ['=== Channel ===',
          f'channel:    {cfg["channel"]}',
          f'source_dir: {cfg["source_dir"]}',
          f'norm:       low={cfg["norm"]["low"]}  high={cfg["norm"]["high"]}',
          f'vae.decoder_ckpt: {cfg.get("vae", {}).get("decoder_ckpt")}', '']
lines += ['=== Flow ===']
for k, v in cfg['flow'].items(): lines.append(f'  {k}: {v}')
lines += ['', '=== Forecast metrics ===']
for p in sorted(glob.glob(f'{CH}/outputs_fc/flow_forecast_*_metrics.json')):
    m = json.load(open(p))
    lines.append(f'\n[{os.path.basename(p)}]  samples={m.get("samples")}  medoid_idx={m.get("medoid_idx")}')
    lines.append(f'  thresholds: {m.get("thresholds")}')
    sbm = m.get('summary_by_mode') or {'(legacy)': m.get('summary', {})}
    for mode, smry in sbm.items():
        lines.append(f'  [{mode}]')
        for k, v in (smry or {}).items():
            lines.append(f'    {k:>10}: {v:.4f}')
    if m.get('crps_mean') is not None:
        lines.append(f'  [crps]')
        lines.append(f'    {"CRPS":>10}: {m["crps_mean"]:.4f}')
with open(os.path.join(SNAP, 'README.txt'), 'w') as f:
    f.write('\n'.join(lines))

zip_path = f'{CH}/results_snapshot_fc_{TS}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in sorted(os.listdir(SNAP)):
        zf.write(os.path.join(SNAP, fn), arcname=os.path.join(os.path.basename(SNAP), fn))
print('Snapshot:', SNAP)
print('Zip:     ', zip_path)